In [13]:
import openai
from qdrant_client import QdrantClient
from dotenv import load_dotenv
from langsmith import traceable, get_current_run_tree
import instructor
from pydantic import BaseModel, Field
from qdrant_client import models
from qdrant_client.models import VectorParams, Filter, FieldCondition, MatchValue, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery


In [2]:
from dotenv import load_dotenv
load_dotenv("../../.env")

True

### Retrieval

In [5]:
qdrant_client = QdrantClient(url="http://localhost:6333")
collection_name="Amazon-shopping-collection-01-hybrid-search"

In [14]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(input=text, model=model)
    return response.data[0].embedding

In [15]:
def retrieve_context_weighted(query, k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name=collection_name, 
        prefetch = [
            Prefetch(query = query_embedding, using = "text-embedding-3-small", limit = 20),
            Prefetch(query = Document(text=query, model = "qdrant/bm25"), using = "bm25", limit = 20)
        ],
        query=models.RrfQuery(rrf = models.Rrf(weights=[3,1])),
        limit=k)

    context_ids = []
    context = []
    scores = []
    ratings = []

    print(results)
    for pt in results.points:
        context_ids.append(pt.payload['parent_asin'])
        context.append(pt.payload['preprocessed_description'])
        scores.append(pt.score)
        ratings.append(pt.payload['average_rating'])
        
    return {
        'retrieved_context_ids': context_ids,
        'retrieved_context': context,
        'similarity_scores': scores,
        'retrieved_context_ratings': ratings
    }

In [16]:
results = retrieve_context_weighted("do you have any 4k webcams ?", k = 20)
results

points=[ScoredPoint(id=826, version=4, score=1.0, payload={'preprocessed_description': 'DEPSTECH Webcam with Microphone, 2K HD Web Camera with Auto Light Correction, Plug and Play USB Webcam and 90° View Angle for Desktop/Streaming/Video Calling Recording/Meeting/Online Teaching【2K QHD Webcam】The HD camera with a 1/2.9”CMOS image sensor delivers sharp and crystal clear video up to a 2560 x 1440 resolution video at a fluid 30 FPS, specially suitable for professional live streaming or recording like detailed makeup tutorials or other online teaching. 【Webcam with Microphone】This web camera built-in microphone with automatic noise reduction make the recording sound clearer, more natural under some noise environments, providing better sound recognition and smooth transmission. 【Smooth Live Streaming Webcam】With automatic low light correction tech, the USB camera captures excellent 2K QHD video at a widescreen angle of up to 90 degrees, even in dimly environment. It also support faster data

{'retrieved_context_ids': ['B08CMV4GF6',
  'B082HQJ1JX',
  'B087986YPY',
  'B002XJG71M',
  'B08R9GPKRH',
  'B0B7QT2KXY',
  'B09YRZ8VDV',
  'B07SFFT6TQ',
  'B01M4J53CO',
  'B07M9Q2PL4',
  'B08XXQCBSZ',
  'B0B7BHY16V',
  'B0B4Z7V3XG',
  'B00PC73NQY',
  'B00HGJQUVG',
  'B016F3M7OM',
  'B07QS5NCB3',
  'B01FFQEP6I',
  'B08PYY2TCC',
  'B08N44NZT3'],
 'retrieved_context': ['DEPSTECH Webcam with Microphone, 2K HD Web Camera with Auto Light Correction, Plug and Play USB Webcam and 90° View Angle for Desktop/Streaming/Video Calling Recording/Meeting/Online Teaching【2K QHD Webcam】The HD camera with a 1/2.9”CMOS image sensor delivers sharp and crystal clear video up to a 2560 x 1440 resolution video at a fluid 30 FPS, specially suitable for professional live streaming or recording like detailed makeup tutorials or other online teaching. 【Webcam with Microphone】This web camera built-in microphone with automatic noise reduction make the recording sound clearer, more natural under some noise environm

### Reranking

In [21]:
import cohere
cohere_client = cohere.ClientV2()

In [19]:
to_rerank = results["retrieved_context"]

In [23]:
response = cohere_client.rerank(
    model="rerank-v4.0-pro", 
    query="do you have bluetooth earphones and webcamera?",
    documents=to_rerank,
    top_n=20,)
response

V2RerankResponse(id='4dd2fefb-41cf-46be-bcf2-e72bd370a48d', results=[V2RerankResponseResultsItem(index=2, relevance_score=0.67542213), V2RerankResponseResultsItem(index=5, relevance_score=0.6509999), V2RerankResponseResultsItem(index=0, relevance_score=0.63304734), V2RerankResponseResultsItem(index=3, relevance_score=0.6054258), V2RerankResponseResultsItem(index=9, relevance_score=0.5210812), V2RerankResponseResultsItem(index=7, relevance_score=0.49765623), V2RerankResponseResultsItem(index=18, relevance_score=0.49765623), V2RerankResponseResultsItem(index=1, relevance_score=0.49375027), V2RerankResponseResultsItem(index=6, relevance_score=0.49375027), V2RerankResponseResultsItem(index=10, relevance_score=0.48594117), V2RerankResponseResultsItem(index=15, relevance_score=0.48594117), V2RerankResponseResultsItem(index=4, relevance_score=0.48203897), V2RerankResponseResultsItem(index=11, relevance_score=0.4664567), V2RerankResponseResultsItem(index=19, relevance_score=0.45868814), V2Rera

In [24]:
response.results

[V2RerankResponseResultsItem(index=2, relevance_score=0.67542213),
 V2RerankResponseResultsItem(index=5, relevance_score=0.6509999),
 V2RerankResponseResultsItem(index=0, relevance_score=0.63304734),
 V2RerankResponseResultsItem(index=3, relevance_score=0.6054258),
 V2RerankResponseResultsItem(index=9, relevance_score=0.5210812),
 V2RerankResponseResultsItem(index=7, relevance_score=0.49765623),
 V2RerankResponseResultsItem(index=18, relevance_score=0.49765623),
 V2RerankResponseResultsItem(index=1, relevance_score=0.49375027),
 V2RerankResponseResultsItem(index=6, relevance_score=0.49375027),
 V2RerankResponseResultsItem(index=10, relevance_score=0.48594117),
 V2RerankResponseResultsItem(index=15, relevance_score=0.48594117),
 V2RerankResponseResultsItem(index=4, relevance_score=0.48203897),
 V2RerankResponseResultsItem(index=11, relevance_score=0.4664567),
 V2RerankResponseResultsItem(index=19, relevance_score=0.45868814),
 V2RerankResponseResultsItem(index=12, relevance_score=0.4548

In [25]:
reranked_reults = [to_rerank[result.index] for result in response.results]
reranked_reults

['Webcam 1080P Full HD Auto Focus Web Camera with Microphone Widescreen USB Computer Camera for PC Laptop Desktop Mac Streaming Video Calling Recording Video Conference Online Teaching Business GamingBuilt-in digital stereo MIC with automatic noise reduction and echo-canceling technology makes the sound purer, clearer and more natural. It can pick up your voice even at 10ft distance. Everyone hears the real you. 5 Million pixels auto focus: Getting rid of tedious focusing process. Our web camera is with 1080P high-definition camera and true color images. Plug And Play: USB 2.0 connector, no need to download or install complicated driver software. It means you needn’t set up any specific driver or software. Our Webcam can plug and play, easy and convenient. Fits for multiple operations including Windows 2000/XP/7/8/10 and above, Mac OS, Android 5.0 or higher version. Works when you need HD video,clear and smooth video conference and video meeting. Works with many video conferencing soft